# Demostración: inferencia con los pesos publicados

Descarga los pesos entrenados y una muestra de cinco objetos de prueba, y sintetiza la órbita completa de un objeto a partir de una sola vista. No requiere entrenar ni descargar el conjunto de datos completo. Funciona con GPU o CPU.

In [ ]:
# 1. Código y dependencias
import os
REPO_URL = "https://github.com/RonaldPort/clifford-nvs.git"
RELEASE_FOLDER_ID = "1TXojt_WquDOaJTpMon-T-5yG6f7GzgNb"   # carpeta pública con pesos y muestra
if not os.path.isdir("/content/clifford-nvs"):
    !git clone -q {REPO_URL} /content/clifford-nvs
%cd /content/clifford-nvs
!pip -q install -r requirements.txt

In [ ]:
# 2. Pesos y muestra
import gdown
if not os.path.isdir("release"):
    gdown.download_folder(id=RELEASE_FOLDER_ID, output="release", quiet=True)
print(sorted(os.listdir("release")))

In [ ]:
# 3. Carga de modelos
import numpy as np, torch
from src.data import load_split, to_float_images
from src.geometry import relative_pose, rotation_angle_deg
from src.metrics import psnr
from src.models import build_model

device = "cuda" if torch.cuda.is_available() else "cpu"
models = {}
for f in sorted(os.listdir("release")):
    if f.endswith(".pt"):
        ck = torch.load(f"release/{f}", map_location=device, weights_only=False)
        m = build_model(ck["cfg"]["model"]).to(device).eval()
        m.load_state_dict(ck["model"])
        models[ck["cfg"]["label"]] = m
sample = load_split("release/demo_sample.npz")
print("Modelos:", list(models), "| objetos en la muestra:", len(sample["obj_offset"]))

In [ ]:
# 4. Órbita completa desde la vista 64 de un objeto
OBJ, SRC_VIEW = 0, 64
off, cnt = sample["obj_offset"][OBJ].item(), sample["obj_count"][OBJ].item()
src_i = torch.full((cnt,), off + SRC_VIEW)
tgt_i = torch.arange(off, off + cnt)
T = relative_pose(sample["poses"][src_i], sample["poses"][tgt_i])
angles = rotation_angle_deg(T[:, :3, :3]).numpy()
target = to_float_images(sample["images"][tgt_i])

preds, scores = {}, {}
with torch.no_grad():
    src = to_float_images(sample["images"][src_i]).to(device)
    for name, m in models.items():
        p = torch.cat([m(src[k:k + 64], T[k:k + 64].to(device)).float().cpu() for k in range(0, cnt, 64)])
        preds[name], scores[name] = p, psnr(p.clamp(0, 1), target).numpy()
        print(f"{name:20s} PSNR medio de la órbita: {scores[name].mean():.2f} dB")

In [ ]:
# 5. PSNR frente al ángulo relativo
import matplotlib.pyplot as plt
order = np.argsort(angles)
plt.figure(figsize=(6, 3))
for name, s in scores.items():
    plt.plot(angles[order], s[order], ".", ms=3, label=name)
plt.xlabel("Ángulo relativo fuente–objetivo (°)"); plt.ylabel("PSNR (dB)")
plt.legend(frameon=False); plt.tight_layout(); plt.show()

In [ ]:
# 6. Animación: fuente | objetivo | predicciones
from PIL import Image as PILImage
from IPython.display import Image, display

def to_uint8(x):
    return (x.clamp(0, 1).permute(0, 2, 3, 1).numpy() * 255).astype(np.uint8)

src_img = sample["images"][off + SRC_VIEW].numpy()
panels = [to_uint8(target)] + [to_uint8(p) for p in preds.values()]
frames = [PILImage.fromarray(np.concatenate([src_img] + [p[k] for p in panels], 1)).resize(
    (64 * (len(panels) + 1) * 3, 64 * 3), PILImage.NEAREST) for k in range(0, cnt, 2)]
frames[0].save("orbita.gif", save_all=True, append_images=frames[1:], duration=60, loop=0)
print("Columnas: fuente | objetivo | " + " | ".join(preds))
display(Image("orbita.gif"))